<a href="https://colab.research.google.com/github/jaker-07/open_wake2.0/blob/main/OWW_Oct20261.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Training your own openWakeWord models


***NOTE (2026-04-11):*** If you get error messages saying that the "runtime needs to be restarted" because packages have already been imported, select restart and run the notebook from the top again.

**Quick-start:** If you just want to train a basic custom model for openWakeWord!

Follow the instructions for Step 1 below. Each time you change the wake word, click the play icon to the left of the title to generate a sample and make sure it sounds correct. The first time it takes a few minutes but subsequent runs will be quick.

Once you're satisfied with the pronounciation, go to the "Runtime" dropdown menu in the upper left of the page, and select "run all". Keep the tab open but feel free to do something else. After ~1 hour, your custom model will be ready and will automatically be downloaded to your computer!

If you are a Home Assistant user with the openWakeWord add-on, follow the instructions [here](https://github.com/home-assistant/addons/blob/master/openwakeword/DOCS.md#custom-wake-word-models) to install and enable your custom model.

---

If you are interested in learning more about the custom model training process (and increasing the accuracy of your custom models), read through each step in this notebook and try experimenting with different training parameters. If you have any questions or problems, feel free to start a discussion at the openWakeWord [repo](https://github.com/dscripka/openWakeWord/discussions).

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 1. Test Example Training Clip Generation
# @markdown Type your wake word below and run the cell to hear it.
# @markdown - Spell it phonetically with underscores if needed ("hey siri" --> "hey_seer_e").
# @markdown - Spell out numbers ("2" --> "two"). No punctuation except "?" and "!".

import os
import sys
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    !cd piper-sample-generator && git checkout 213d4d5

    !pip install piper-tts piper-phonemize-cross
    !pip install webrtcvad
    !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121

target_word = 'hey_karl' # @param {type:"string"}

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

def text_to_speech(text):
    generate_samples(text = text,
                max_samples=1,
                length_scales=[1.1],
                noise_scales=[0.7], noise_scale_ws = [0.7],
                output_dir = './', batch_size=1, auto_reduce_batch_size=True,
                file_names=["test_generation.wav"]
                )

text_to_speech(target_word)
Audio("test_generation.wav", autoplay=True)


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 2a. Install training dependencies
# @markdown Run once. **The runtime restarts automatically at the end** - a
# @markdown "Your session crashed" / "restarting" message is expected. Then run 2b.

import os, locale
locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"

if not os.path.exists("./openwakeword"):
    !git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword --no-deps

# One pip call so every pin is resolved together.
# - numpy/protobuf/setuptools stay at Colab's versions (librosa's numba needs numpy<2.1)
# - onnx2tf 1.28.8 is the last release that doesn't force numpy 2.2.6 / protobuf 7
# - ai_edge_litert 2.1.2: 1.4.0 crashes next to Colab's TensorFlow 2.20 during conversion
!pip install -q "numpy==2.0.2" "protobuf==5.29.6" "setuptools<81" \
    mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 \
    audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
    onnxruntime==1.22.1 ai_edge_litert==2.1.2 onnxsim onnx==1.19.1 onnx2tf==1.28.8 \
    onnx_graphsurgeon sng4onnx pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19

# openWakeWord feature models
models_dir = "./openwakeword/openwakeword/resources/models"
os.makedirs(models_dir, exist_ok=True)
for f in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    if not os.path.exists(f"{models_dir}/{f}"):
        !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f} -O {models_dir}/{f}

# Restart the runtime so the new package versions load cleanly
os.kill(os.getpid(), 9)

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 2b. Download Data
# @markdown Downloads room impulse responses, background noise/music and the
# @markdown pre-computed openWakeWord features (~16 GB). Safe to re-run: finished
# @markdown parts are skipped.
# @markdown
# @markdown **Note:** this data has mixed licenses - models trained with it are for
# @markdown **non-commercial** personal use only.

import os, sys, io
from pathlib import Path
import numpy as np
import scipy, scipy.io.wavfile
import soundfile as sf
import librosa
import pyarrow as pa, pyarrow.parquet as pq
import datasets
from tqdm import tqdm

def write_wav(path, audio):
    scipy.io.wavfile.write(path, 16000, (np.clip(audio, -1, 1)*32767).astype(np.int16))

## MIT room impulse responses (~2 min)
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
if not os.listdir(output_dir):
    if not os.path.exists("MIT_environmental_impulse_responses"):
        !git lfs install
        !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())
    for row in tqdm(rir_dataset, desc="RIRs"):
        write_wav(os.path.join(output_dir, row['audio']['path'].split('/')[-1]), row['audio']['array'])
print(len(os.listdir(output_dir)), "RIR files in", output_dir)

## AudioSet background noise (~3 min)
# The dataset owner converted AudioSet to Parquet, so the old .tar 404s on the main branch.
# First try the .tar from the last revision before that change, then fall back to Parquet.
def parquet_to_wavs(pf, output_dir):
    f = pq.ParquetFile(pf)
    col = next(fl.name for fl in f.schema_arrow
               if pa.types.is_struct(fl.type) and "bytes" in [c.name for c in fl.type])
    n = 0
    for batch in tqdm(f.iter_batches(batch_size=32, columns=[col]), total=f.metadata.num_row_groups, desc="AudioSet"):
        for a in batch.column(0).to_pylist():
            if not a or not a.get("bytes"):
                continue
            try:
                y, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32", always_2d=True)
            except Exception:
                continue
            y = librosa.resample(y.mean(axis=1), orig_sr=sr, target_sr=16000)
            name = os.path.splitext(os.path.basename(a.get("path") or f"clip_{n}"))[0] + ".wav"
            write_wav(os.path.join(output_dir, name), y)
            n += 1
    return n

output_dir = "./audioset_16k"
os.makedirs("audioset", exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
fname = "bal_train09.tar"
OLD_REV = "196c0900867eff791b8f4d4be57db277e9a5b131"
if not os.listdir(output_dir):
    if not os.path.exists("audioset/audio"):
        !rm -f audioset/{fname}
        !wget -q -O audioset/{fname} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/{OLD_REV}/data/{fname}
        if os.path.getsize(f"audioset/{fname}") > 100_000_000:
            !cd audioset && tar -xf {fname}
    flacs = [str(i) for i in Path("audioset/audio").glob("**/*.flac")]
    if flacs:
        audioset_dataset = datasets.Dataset.from_dict({"audio": flacs}).cast_column("audio", datasets.Audio(sampling_rate=16000))
        for row in tqdm(audioset_dataset, desc="AudioSet"):
            write_wav(os.path.join(output_dir, row['audio']['path'].split('/')[-1].replace(".flac", ".wav")), row['audio']['array'])
    else:
        print("Old AudioSet .tar not available - falling back to the Parquet version")
        try:
            from huggingface_hub import list_repo_files, hf_hub_download
            pq_files = sorted(f for f in list_repo_files("agkphysics/AudioSet", repo_type="dataset") if f.endswith(".parquet"))
            bal = [f for f in pq_files if "bal" in f.lower() and "unbal" not in f.lower()] or pq_files
            print("Using", bal[0])
            parquet_to_wavs(hf_hub_download("agkphysics/AudioSet", bal[0], repo_type="dataset", local_dir="audioset"), output_dir)
        except Exception as e:
            print("AudioSet download failed, training will use FMA music only for background noise:", repr(e))
print(len(os.listdir(output_dir)), "AudioSet clips in", output_dir)

## Free Music Archive (1 hour of clips)
output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
if not os.listdir(output_dir):
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
    n_hours = 1  # increase for full-scale training
    for _ in tqdm(range(n_hours*3600//30), desc="FMA"):  # FMA clips are 30 s each
        row = next(fma_dataset)
        write_wav(os.path.join(output_dir, row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")), row['audio']['array'])
print(len(os.listdir(output_dir)), "FMA clips in", output_dir)

## Pre-computed openWakeWord features (-c resumes partial downloads and skips finished ones)
!wget -c -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -c -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Done. Feature files:", {f: os.path.getsize(f) for f in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"] if os.path.exists(f)})

In [ ]:
%%writefile openwakeword/openwakeword/gpu_features.py
# GPU feature extraction for openWakeWord training.
#
# openWakeWord computes its training features with two small ONNX models (melspectrogram + embedding).
# On Colab the installed onnxruntime is CPU-only, so this step ran on one CPU thread. This module runs
# the same two ONNX graphs with PyTorch on the GPU.
#
# Safety: the first batch is computed both ways (GPU and openWakeWord's own CPU code) and compared. Every GPU
# batch is checked for shape and NaN/inf, and one clip every few batches (plus the last batch) is re-done on the
# CPU. On any mismatch or error, the GPU batches since the last passed check and the rest of the file are
# computed with the CPU code. If the GPU can't be used at all, this is exactly openWakeWord's CPU code.

import logging
import os

import numpy as np
import onnx
import torch
import torch.nn.functional as F
from numpy.lib.format import open_memmap
from onnx import numpy_helper
from tqdm import tqdm

from openwakeword.utils import AudioFeatures

_MODELS = os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources", "models")
_TOLERANCE = 1e-3     # max allowed |GPU - CPU| difference, relative to the feature scale
_RECHECK_EVERY = 5    # batches between spot checks (1 clip re-done on the CPU)


class _TorchGraph:
    """Runs one of openWakeWord's feature ONNX graphs with PyTorch (only the ops those graphs use)."""

    def __init__(self, path, device):
        model = onnx.load(path)
        to_t = lambda a: torch.from_numpy(np.array(a, copy=True)).to(device)
        self.values = {i.name: to_t(numpy_helper.to_array(i)) for i in model.graph.initializer}
        self.nodes = []
        for n in model.graph.node:
            attrs = {a.name: onnx.helper.get_attribute_value(a) for a in n.attribute}
            if n.op_type == "Constant":
                self.values[n.output[0]] = to_t(numpy_helper.to_array(attrs["value"]))
            else:
                self.nodes.append((n.op_type, list(n.input), n.output[0], attrs))
        self.input_name = model.graph.input[0].name
        self.output_name = model.graph.output[0].name
        # free each intermediate after the last node that reads it (otherwise every activation stays in memory)
        last_use = {k: n for n, (_, names, _, _) in enumerate(self.nodes) for k in names}
        self.free_after = [{k for k in names if k and last_use[k] == n and k not in self.values}
                           for n, (_, names, _, _) in enumerate(self.nodes)]

    def __call__(self, x):
        v = dict(self.values)
        v[self.input_name] = x
        for n, (op, names, out, a) in enumerate(self.nodes):
            i = [v[k] if k else None for k in names]  # keep positions: '' marks an omitted optional input
            for k in self.free_after[n]:
                del v[k]
            if op == "Conv":
                nd = i[1].dim() - 2
                pads = list(a.get("pads", [0] * 2 * nd))
                y = i[0]
                if pads[:nd] != pads[nd:]:
                    spec = []
                    for d in reversed(range(nd)):
                        spec += [pads[d], pads[nd + d]]
                    y, padding = F.pad(y, spec), 0
                else:
                    padding = tuple(pads[:nd])
                conv = F.conv1d if nd == 1 else F.conv2d
                y = conv(y, i[1], i[2] if len(i) > 2 else None, stride=tuple(a.get("strides", [1] * nd)),
                         padding=padding, dilation=tuple(a.get("dilations", [1] * nd)), groups=a.get("group", 1))
            elif op == "MaxPool":
                pads = list(a.get("pads", [0, 0, 0, 0]))
                if pads[:2] != pads[2:] or a.get("ceil_mode", 0) or any(d != 1 for d in a.get("dilations", [1, 1])):
                    raise NotImplementedError("MaxPool variant")
                y = F.max_pool2d(i[0], tuple(a["kernel_shape"]), tuple(a.get("strides", [1, 1])),
                                 padding=tuple(pads[:2]))
            elif op == "LeakyRelu":
                y = F.leaky_relu(i[0], a.get("alpha", 0.01))
            elif op == "Max":
                y = i[0]
                for t in i[1:]:
                    y = torch.maximum(y, t)
            elif op == "Reshape":
                shape = [int(s) for s in i[1].tolist()]
                shape = [i[0].shape[k] if s == 0 else s for k, s in enumerate(shape)]
                y = i[0].reshape(shape)
            elif op == "Unsqueeze":
                y = i[0]
                for ax in sorted(int(t) for t in i[1].reshape(-1).tolist()):
                    y = y.unsqueeze(ax)
            elif op == "Transpose":
                y = i[0].permute(*a["perm"])
            elif op == "Pow":
                y = torch.pow(i[0], i[1])
            elif op == "Add":
                y = i[0] + i[1]
            elif op == "Sub":
                y = i[0] - i[1]
            elif op == "Mul":
                y = i[0] * i[1]
            elif op == "Div":
                y = i[0] / i[1]
            elif op == "MatMul":
                y = torch.matmul(i[0], i[1])
            elif op == "Log":
                y = torch.log(i[0])
            elif op == "Clip":
                y = i[0]
                if len(i) > 1 and i[1] is not None:
                    y = torch.maximum(y, i[1])
                if len(i) > 2 and i[2] is not None:
                    y = torch.minimum(y, i[2])
            elif op == "ReduceMax":
                # The ONNX graph takes one max over the whole input. openWakeWord's CPU path feeds it one
                # clip at a time, so take the max per clip to get exactly the CPU result for a batch.
                if "axes" in a or len(i) > 1 or a.get("keepdims", 1) != 0:
                    raise NotImplementedError("ReduceMax variant")
                y = i[0].amax(dim=tuple(range(1, i[0].dim())), keepdim=True)
            elif op == "Cast":
                if a.get("to") != onnx.TensorProto.FLOAT:
                    raise NotImplementedError("Cast variant")
                y = i[0].float()
            else:
                raise NotImplementedError(f"ONNX op {op}")
            v[out] = y
        return v[self.output_name]


class TorchAudioFeatures:
    """Drop-in replacement for AudioFeatures.embed_clips/get_embedding_shape that runs on a torch device."""

    def __init__(self, device):
        self.device = torch.device(device)
        self.melspec = _TorchGraph(os.path.join(_MODELS, "melspectrogram.onnx"), self.device)
        self.embedding = _TorchGraph(os.path.join(_MODELS, "embedding_model.onnx"), self.device)

    @torch.no_grad()
    def embed_clips(self, x, batch_size=128, ncpu=1, max_windows=512):
        x = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(self.device)
        spec = self.melspec(x)                      # (N, 1, frames, 32)
        spec = spec[:, 0] / 10 + 2                  # openWakeWord's melspec transform
        n_clips = spec.shape[0]
        windows = spec.unfold(1, 76, 8).permute(0, 1, 3, 2)  # (N, n_windows, 76, 32), step 8 like openWakeWord
        n_windows = windows.shape[1]
        windows = windows.reshape(-1, 76, 32, 1)
        out = torch.cat([self.embedding(windows[k:k + max_windows]) for k in range(0, windows.shape[0], max_windows)])
        return out.reshape(n_clips, n_windows, 96).cpu().numpy()

    def get_embedding_shape(self, audio_length, sr=16000):
        x = (np.random.uniform(-1, 1, (1, int(audio_length * sr))) * 32767).astype(np.int16)
        return self.embed_clips(x).shape[1:]


def _torch_device():
    forced = os.environ.get("OWW_TORCH_FEATURES_DEVICE")  # for testing on machines without a GPU
    if forced:
        return forced
    return "cuda" if torch.cuda.is_available() else None


def _close_enough(a, b):
    return a.shape == b.shape and float(np.abs(a - b).max()) <= _TOLERANCE * max(1.0, float(np.abs(b).max()))


def compute_features_from_generator(generator, n_total, clip_duration, output_file, device="cpu", ncpu=1):
    """Same contract and output file as openwakeword.utils.compute_features_from_generator."""
    from openwakeword.data import trim_mmap

    cpu = AudioFeatures(device="cpu", ncpu=1)
    fast = None
    dev = _torch_device()
    if dev is not None:
        try:
            if dev.startswith("cuda"):
                # full float32 precision so results match the CPU (TF32 would add ~1e-3 error)
                torch.backends.cuda.matmul.allow_tf32 = False
                torch.backends.cudnn.allow_tf32 = False
            fast = TorchAudioFeatures(dev)
        except Exception as e:
            logging.warning(f"GPU features unavailable ({e!r}); using the CPU")

    n_feature_cols = cpu.get_embedding_shape(clip_duration / 16000)
    output_shape = (n_total, n_feature_cols[0], n_feature_cols[1])
    fp = open_memmap(output_file, mode='w+', dtype=np.float32, shape=output_shape)

    row_counter = 0
    audio_data = next(generator)
    batch_size = audio_data.shape[0]
    if batch_size > n_total:
        raise ValueError(f"The value of 'n_total' ({n_total}) is less than the batch size ({batch_size})."
                         " Please increase 'n_total' to be >= batch size.")

    features = cpu.embed_clips(audio_data, batch_size=batch_size)
    if fast is not None:
        try:
            if fast.get_embedding_shape(clip_duration / 16000) != tuple(n_feature_cols) or \
                    not _close_enough(fast.embed_clips(audio_data), features):
                raise ValueError("GPU features differ from the CPU features")
            logging.info(f"Computing features on {dev} (verified against the CPU)")
        except Exception as e:
            logging.warning(f"GPU features check failed ({e!r}); using the CPU")
            fast = None

    fp[row_counter:row_counter + features.shape[0], :, :] = features
    row_counter += features.shape[0]
    fp.flush()

    def spot_check(audio, features, k):
        if not _close_enough(features[k:k + 1], cpu.embed_clips(audio[k:k + 1], batch_size=1)):
            raise ValueError("GPU features drifted from the CPU features")

    def redo_on_cpu(batches, why):
        # rows written by the GPU since the last passed spot check are not trusted any more: recompute them
        logging.warning(f"GPU features failed ({why!r}); redoing {len(batches)} batch(es) and the rest of this file on the CPU")
        for start, audio in batches:
            fp[start:start + audio.shape[0], :, :] = cpu.embed_clips(audio, batch_size=batch_size, ncpu=ncpu)
        fp.flush()

    unverified = []  # (first row, audio) of GPU batches written since the last passed spot check
    for batch_ndx, audio_data in enumerate(tqdm(generator, total=n_total // batch_size, desc="Computing features")):
        if row_counter >= n_total:
            break

        features = None
        if fast is not None:
            try:
                features = fast.embed_clips(audio_data)
                if features.shape != (audio_data.shape[0], *n_feature_cols) or not np.isfinite(features).all():
                    raise ValueError(f"GPU features have shape {features.shape} or non-finite values")
                if (batch_ndx + 1) % _RECHECK_EVERY == 0:
                    spot_check(audio_data, features, (batch_ndx // _RECHECK_EVERY) % audio_data.shape[0])
                    unverified = []
            except Exception as e:
                redo_on_cpu(unverified, e)
                fast, features, unverified = None, None, []
        if features is None:
            features = cpu.embed_clips(audio_data, batch_size=batch_size, ncpu=ncpu)

        if row_counter + features.shape[0] > n_total:
            features = features[0:n_total - row_counter]
        if fast is not None and (batch_ndx + 1) % _RECHECK_EVERY != 0:
            unverified.append((row_counter, audio_data[:features.shape[0]]))

        fp[row_counter:row_counter + features.shape[0], :, :] = features
        row_counter += features.shape[0]
        fp.flush()

    if unverified:  # one last spot check (on the last batch, which is often a different size)
        start, audio = unverified[-1]
        try:
            spot_check(audio, np.asarray(fp[start:start + audio.shape[0]]), audio.shape[0] - 1)
        except Exception as e:
            redo_on_cpu(unverified, e)
    trim_mmap(output_file)

Overwriting openwakeword/openwakeword/gpu_features.py


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 3. Train the Model
# @markdown - `target_word`: must be the same spelling you tested in step 1.
# @markdown - `adversarial_phrase`: the same wake word written as **real English
# @markdown words** (e.g. "hey karl"). It is only used to generate similar-sounding
# @markdown *negative* examples. (openWakeWord used to look up made-up words with a
# @markdown downloadable model, but that download has been taken offline.)
# @markdown - `number_of_examples`: 20,000+ recommended (30,000-50,000 is best).
# @markdown - `number_of_training_steps`: 30,000+ recommended (50,000 is openWakeWord's default).
# @markdown - `false_activation_penalty`: higher = fewer false activations, but may miss
# @markdown unclear/noisy wake words. 1,000-1,500 recommended (openWakeWord automatically raises it
# @markdown 4x during the final training phases).
# @markdown
# @markdown **Checkpoints:** each finished stage (the generated clips, then each feature file) is saved to
# @markdown Google Drive in `MyDrive/hey_karl/checkpoints/`. If Colab disconnects, run cells 1, 2a, 2b
# @markdown (and 2c) again, then this cell with the **same settings** - it continues from the last saved stage.
# @markdown Changing `number_of_training_steps`, `false_activation_penalty` or `layer_size` also reuses
# @markdown the saved clips and features, so a re-train only takes the training time.
# @markdown
# @markdown When it finishes, the `.onnx` and `.tflite` files download automatically and are also saved
# @markdown to `MyDrive/hey_karl/`.

import os, sys, yaml, onnx, pronouncing, shutil, glob, json, hashlib, subprocess
import numpy as np
from google.colab import drive
drive.mount("/content/drive")  # finished models and checkpoints are saved here so a disconnect can't lose them
drive_dir = "/content/drive/MyDrive/hey_karl/"

target_word = 'hey_karl' # @param {type:"string"}
adversarial_phrase = 'hey carl' # @param {type:"string"}
number_of_examples = 50000 # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 50000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 900  # @param {type:"slider", min:100, max:5000, step:50}
augmentation_rounds = 2  # @param {type:"slider", min:1, max:3, step:1}
layer_size = 64  # @param [32, 64, 128] {type:"raw"}

# ---- Pre-flight checks: fail now, not hours into training ----
missing = [w for w in adversarial_phrase.lower().split() if not pronouncing.phones_for_word(w)]
assert not missing, f"These words aren't in the pronunciation dictionary: {missing}. Change adversarial_phrase to common English words."
for f in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"]:
    assert os.path.exists(f), f"{f} is missing - run cell 2b first"
assert os.listdir("./mit_rirs"), "./mit_rirs is empty - run cell 2b first"
background_paths = [p for p in ["./audioset_16k", "./fma"] if os.path.isdir(p) and os.listdir(p)]
assert background_paths, "No background audio found - run cell 2b first"
import torch
print("Training on", "GPU: " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow - see instructions)")
print("Background audio:", background_paths)

# ---- Patch openWakeWord (safe to re-run) ----
# 1) build negative examples from `adversarial_phrase` (the phonemizer download for made-up words is gone)
# 2) don't run the built-in TFLite conversion: its --convert_to_tflite flag defaults to the string "False",
#    which is truthy, so it always runs and crashes on the missing onnx_tf package. We convert below instead.
train_py = "openwakeword/openwakeword/train.py"
src = open(train_py).read()
src = src.replace('input_text=target_phrase,', 'input_text=config.get("adversarial_phrase", target_phrase),')
src = src.replace('if args.convert_to_tflite:', 'if args.convert_to_tflite is True:')
# 3) augmentation_rounds > 1 is silently thrown away upstream: the clip list is repeated, but the feature file is
#    sized (and cut off) at one round's worth of clips. Size it for all rounds.
src = src.replace('n_total=len(os.listdir(', 'n_total=config["augmentation_rounds"]*len(os.listdir(')
# 4) feature computation: onnxruntime here is CPU-only, but on a GPU runtime train.py asks for the GPU and
#    then uses 1 CPU thread. Use openWakeWord's own multi-core CPU path instead (same results, faster).
src = src.replace('device="gpu" if torch.cuda.is_available() else "cpu",', 'device="cpu",')
src = src.replace('ncpu=n_cpus if not torch.cuda.is_available() else 1)', 'ncpu=n_cpus)')
# 5) checkpoints: route feature computation through oww_checkpoint.py (written below). It restores a finished
#    feature file from Google Drive instead of recomputing it, and saves each newly finished file there.
#    The computation itself is gpu_features.py from cell 2c if present, else openWakeWord's CPU code.
ckpt_import = 'from openwakeword.oww_checkpoint import compute_features_from_generator'
for old_import in ['from openwakeword.utils import compute_features_from_generator',
                   'from openwakeword.gpu_features import compute_features_from_generator']:
    src = src.replace(old_import, ckpt_import)
assert src.count('config.get("adversarial_phrase", target_phrase)') == 2 and 'if args.convert_to_tflite is True:' in src \
    and src.count('n_total=config["augmentation_rounds"]*len(os.listdir(') == 4 \
    and src.count('ncpu=n_cpus)') >= 4 and 'device="gpu" if torch.cuda.is_available()' not in src \
    and ckpt_import in src, \
    "train.py has changed upstream; the patch didn't apply"
open(train_py, "w").write(src)

open("openwakeword/openwakeword/oww_checkpoint.py", "w").write('''\
# Written by cell 3. Wraps openWakeWord's feature computation so each finished feature file is saved to
# Google Drive (folder in $OWW_CKPT_DIR) and restored from there after a disconnect instead of recomputed.
import logging, os, shutil
import numpy as np
try:
    from openwakeword.gpu_features import compute_features_from_generator as _compute  # cell 2c (GPU)
except Exception:
    from openwakeword.utils import compute_features_from_generator as _compute

def _complete(path):
    try:
        a = np.load(path, mmap_mode="r")
        return a.ndim == 3 and a.shape[0] > 0 and bool(np.any(a[-1]))  # finished files are trimmed
    except Exception:
        return False

def compute_features_from_generator(generator, n_total, clip_duration, output_file, **kwargs):
    name = os.path.basename(output_file)
    ckpt_dir = os.environ.get("OWW_CKPT_DIR")
    saved = os.path.join(ckpt_dir, name) if ckpt_dir else None
    if not _complete(output_file) and saved and _complete(saved):
        shutil.copyfile(saved, output_file)
        logging.info(f"Checkpoint: restored {name} from Google Drive")
    if _complete(output_file):
        logging.info(f"{name} is already computed - skipping it")
    else:
        _compute(generator, n_total, clip_duration, output_file, **kwargs)
    if saved and not _complete(saved):
        try:
            shutil.copyfile(output_file, saved + ".tmp")
            os.replace(saved + ".tmp", saved)
            logging.info(f"Checkpoint: saved {name} to Google Drive")
        except Exception as e:
            logging.warning(f"Checkpoint: could not save {name} to Google Drive ({e!r})")
''')
print("Feature computation:", ("GPU (cell 2c)" if torch.cuda.is_available() else "CPU (no GPU in this runtime)")
      if os.path.exists("openwakeword/openwakeword/gpu_features.py") else "CPU - run cell 2c to use the GPU")

# 6) openWakeWord's homophone filter for negatives never matches (it compares a string to a list), so
#    phrases that sound exactly like the wake word (e.g. "hay karl") end up labelled as negatives. Drop them.
data_py = "openwakeword/openwakeword/data.py"
src = open(data_py).read()
old = "    adversarial_texts = [i for i in adversarial_texts if i != input_text]\n"
new = ("    _p = lambda t: [pronouncing.phones_for_word(w) for w in t.split()]\n"
       "    _s = lambda ps: None if [] in ps else ' '.join(re.sub(r'\\d', '', p[0]) for p in ps)\n"
       "    _target = _s(_p(input_text))\n"
       "    adversarial_texts = [i for i in adversarial_texts if i != input_text and (_target is None or _s(_p(i)) != _target)]\n")
if new not in src:
    assert src.count(old) == 1, "data.py has changed upstream; the patch didn't apply"
    open(data_py, "w").write(src.replace(old, new))

# 7) piper's out-of-GPU-memory fallback shrinks the batch, then still loops over the full batch -> IndexError.
#    Loop over the clips that were actually generated.
gen_py = "piper-sample-generator/generate_samples.py"
src = open(gen_py).read()
old_l = "for i, clip_phoneme_index in enumerate(clip_indexes_by_batch):"
new_l = "for i, clip_phoneme_index in enumerate(clip_indexes_by_batch[:audio.shape[0]]):"
if new_l not in src:
    assert src.count(old_l) == 1, "generate_samples.py has changed upstream; the patch didn't apply"
    open(gen_py, "w").write(src.replace(old_l, new_l))

# ---- Write the training config ----
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config["target_phrase"] = [target_word]
config["adversarial_phrase"] = adversarial_phrase
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = number_of_examples
config["n_samples_val"] = max(500, number_of_examples//10)
config["steps"] = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = false_activation_penalty
config["background_paths"] = background_paths
config["background_paths_duplication_rate"] = [1]*len(background_paths)
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["augmentation_rounds"] = augmentation_rounds
config["layer_size"] = layer_size
config["tts_batch_size"] = 100  # clips per GPU batch when generating (default 50; negatives use 1/7 of it). Retries smaller if GPU memory runs out (patch 7).
with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

# ---- Local re-runs: clear clips/features that were made with different settings ----
run_dir = f"my_custom_model/{config['model_name']}"
clip_targets = {"positive_train": config["n_samples"], "positive_test": config["n_samples_val"],
                "negative_train": config["n_samples"], "negative_test": config["n_samples_val"]}
feature_files = [f"{run_dir}/{s}_features_{t}.npy" for s in ("positive", "negative") for t in ("train", "test")]
data_key = f"{target_word}|{adversarial_phrase}|{number_of_examples}|{augmentation_rounds}"
stamp_path = f"{run_dir}/data_settings.txt"
old_key = open(stamp_path).read() if os.path.exists(stamp_path) else None
if old_key != data_key:
    if old_key is not None or glob.glob(f"{run_dir}/*.npy"):
        print("Settings changed since the last run here - recomputing features (matching clips are reused)")
    for f in glob.glob(f"{run_dir}/*.npy"):
        os.remove(f)
    if old_key is not None and old_key.split("|")[:2] != data_key.split("|")[:2]:
        for d in clip_targets:
            shutil.rmtree(f"{run_dir}/{d}", ignore_errors=True)
os.makedirs(run_dir, exist_ok=True)
open(stamp_path, "w").write(data_key)

# ---- Checkpoints on Google Drive (one folder per clip/feature settings) ----
def n_clips(sub):
    d = f"{run_dir}/{sub}"
    return len(os.listdir(d)) if os.path.isdir(d) else 0

def clips_ready(sub):  # train.py's own rule for "already generated"
    return n_clips(sub) > 0.95 * clip_targets[sub]

def feature_complete(path):
    try:
        a = np.load(path, mmap_mode="r")
        return a.ndim == 3 and a.shape[0] > 0 and bool(np.any(a[-1]))  # finished files are trimmed
    except Exception:
        return False

ckpt_dir = os.path.join(drive_dir, "checkpoints",
                        f"{config['model_name']}_{hashlib.sha1(data_key.encode()).hexdigest()[:8]}")
try:
    os.makedirs(ckpt_dir, exist_ok=True)
    print("Checkpoint folder:", ckpt_dir)
except Exception as e:
    print(f"WARNING: can't use Google Drive for checkpoints ({e!r}) - training continues without them")
    ckpt_dir = None
os.environ["OWW_CKPT_DIR"] = ckpt_dir or ""  # read by oww_checkpoint.py in the train.py processes

def save_clips(sub):
    tar, marker = f"{ckpt_dir}/{sub}.tar", f"{ckpt_dir}/{sub}.tar.json"
    if not ckpt_dir or os.path.exists(marker):
        return
    try:
        subprocess.run(["tar", "-cf", tar + ".tmp", "-C", run_dir, sub], check=True)
        size = os.path.getsize(tar + ".tmp")
        os.replace(tar + ".tmp", tar)
        json.dump({"files": n_clips(sub), "bytes": size}, open(marker, "w"))
        print(f"Checkpoint: saved {n_clips(sub)} {sub} clips to Google Drive")
    except Exception as e:
        print(f"WARNING: could not save {sub} clips to Google Drive ({e!r})")

def restore_clips(sub):
    tar, marker = f"{ckpt_dir}/{sub}.tar", f"{ckpt_dir}/{sub}.tar.json"
    if not ckpt_dir or clips_ready(sub) or not os.path.exists(marker):
        return
    try:
        info = json.load(open(marker))
        if os.path.getsize(tar) != info["bytes"]:
            raise ValueError("the saved archive is incomplete")
        shutil.rmtree(f"{run_dir}/{sub}", ignore_errors=True)
        subprocess.run(["tar", "-xf", tar, "-C", run_dir], check=True)
        if n_clips(sub) != info["files"]:
            raise ValueError(f"got {n_clips(sub)} clips, expected {info['files']}")
        print(f"Checkpoint: restored {n_clips(sub)} {sub} clips from Google Drive")
    except Exception as e:
        print(f"WARNING: could not restore {sub} clips ({e!r}) - they will be generated again")
        shutil.rmtree(f"{run_dir}/{sub}", ignore_errors=True)
        for p in (marker, tar):  # drop the bad checkpoint so the regenerated clips get saved again
            try:
                os.remove(p)
            except OSError:
                pass

if ckpt_dir:
    for f in feature_files:
        saved = f"{ckpt_dir}/{os.path.basename(f)}"
        if not feature_complete(f) and feature_complete(saved):
            shutil.copyfile(saved, f)
            print(f"Checkpoint: restored {os.path.basename(f)} from Google Drive")
features_done = all(feature_complete(f) for f in feature_files)
for sub in (["positive_test"] if features_done else clip_targets):  # train.py reads a few positive_test clips
    restore_clips(sub)

# ---- Train: generate clips -> augment -> train, skipping stages that are already done ----
onnx_model_path = f"my_custom_model/{config['model_name']}.onnx"
for f in glob.glob(f"my_custom_model/{config['model_name']}*.onnx") + glob.glob(f"my_custom_model/{config['model_name']}*.tflite"):
    os.remove(f)  # never convert/download a model left over from an earlier run

if not features_done or not clips_ready("positive_test"):
    !{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips
    not_ready = [s for s in clip_targets if not clips_ready(s)]
    assert not not_ready, f"Clip generation did not finish ({not_ready}) - scroll up to the first error"
    for sub in clip_targets:
        save_clips(sub)
else:
    print("Clips and features restored - skipping clip generation")

if not features_done:
    !{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips --overwrite
    unfinished = [os.path.basename(f) for f in feature_files if not feature_complete(f)]
    assert not unfinished, f"Feature computation did not finish ({unfinished}) - scroll up to the first error"
else:
    print("All features restored - skipping augmentation")

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model
assert os.path.exists(onnx_model_path), "Training did not produce a model - scroll up to the first error"
os.makedirs(drive_dir, exist_ok=True)
shutil.copy(onnx_model_path, drive_dir)  # save to Google Drive right away
print("Saved", onnx_model_path, "to Google Drive ->", drive_dir)

# ---- Convert ONNX -> TFLite with onnx2tf ----
tflite_path = f"my_custom_model/{config['model_name']}.tflite"
float32_path = f"my_custom_model/{config['model_name']}_float32.tflite"
input_name = onnx.load(onnx_model_path).graph.input[0].name     # e.g. "onnx::Flatten_0"
kat_name = input_name.replace(":", "__")                       # onnx2tf's name for it, e.g. "onnx____Flatten_0"
!onnx2tf -i {onnx_model_path} -o my_custom_model/ -kat {kat_name}
assert os.path.exists(float32_path), "onnx2tf did not produce a .tflite file - scroll up for the error"
os.replace(float32_path, tflite_path)

# ---- Check the .tflite gives the same answers as the .onnx ----
import onnxruntime as ort
from ai_edge_litert.interpreter import Interpreter
sess = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
interp = Interpreter(model_path=tflite_path); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
assert list(inp["shape"]) == list(sess.get_inputs()[0].shape), f"TFLite input shape {inp['shape']} != ONNX {sess.get_inputs()[0].shape}"
x = np.random.rand(*inp["shape"]).astype(np.float32)
interp.set_tensor(inp["index"], x); interp.invoke()
diff = np.abs(interp.get_tensor(out["index"]).ravel() - sess.run(None, {sess.get_inputs()[0].name: x})[0].ravel()).max()
assert diff < 1e-3, f"TFLite and ONNX outputs differ by {diff}"
print(f"Model OK: input shape {[int(d) for d in inp['shape']]}, ONNX vs TFLite max difference {diff:.1e}")

# ---- Save to Drive + download ----
shutil.copy(tflite_path, drive_dir)
print("Saved", tflite_path, "to Google Drive ->", drive_dir)
from google.colab import files
files.download(onnx_model_path)
files.download(tflite_path)

In [ ]:
# @title  { display-mode: "form" }
# @markdown # Resume: train hey_karl from the Google Drive checkpoint
# @markdown Run this ONE cell on a fresh runtime (runtime version 2026.07, A100 or H100). No other cells needed.
# @markdown It installs what training needs, downloads the 2 big feature files, restores your saved
# @markdown features from Google Drive, trains, converts to TFLite, saves both models to Drive and downloads them.
# @markdown No clip generation, no feature computation.

import os, sys, glob, json, shutil, subprocess, locale
locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"
from google.colab import drive, files

number_of_training_steps = 50000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 900  # @param {type:"slider", min:100, max:5000, step:50}
layer_size = 64  # @param {type:"slider", min:32, max:128, step:32}

# ---- Find the checkpoint on Google Drive ----
drive.mount("/content/drive")
drive_dir = "/content/drive/MyDrive/hey_karl/"
feature_names = [f"{s}_features_{t}.npy" for s in ("positive", "negative") for t in ("train", "test")]
needed = feature_names + ["positive_test.tar", "positive_test.tar.json"]
ckpt_dir = os.path.join(drive_dir, "checkpoints", "hey_karl_7f68b4bd")  # the "Checkpoint folder:" from your last run
if not all(os.path.exists(os.path.join(ckpt_dir, f)) for f in needed):
    found = sorted((d for d in glob.glob(os.path.join(drive_dir, "checkpoints", "hey_karl_*"))
                    if all(os.path.exists(os.path.join(d, f)) for f in needed)), key=os.path.getmtime)
    assert found, f"No checkpoint folder in {drive_dir}checkpoints has all of {needed}"
    ckpt_dir = found[-1]
print("Using checkpoint folder:", ckpt_dir)

# ---- Install (same commands and versions as cells 1 and 2a) ----
if not os.path.exists("./piper-sample-generator"):  # train.py imports it even when it only trains
    !git clone https://github.com/rhasspy/piper-sample-generator
    !cd piper-sample-generator && git checkout 213d4d5
!pip install -q piper-tts piper-phonemize-cross
!pip install -q webrtcvad
!pip install -q torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121
if not os.path.exists("./openwakeword"):
    !git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword --no-deps
!pip install -q "numpy==2.0.2" "protobuf==5.29.6" "setuptools<81" \
    mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 \
    audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
    onnxruntime==1.22.1 ai_edge_litert==2.1.2 onnxsim onnx==1.19.1 onnx2tf==1.28.8 \
    onnx_graphsurgeon sng4onnx pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19
models_dir = "./openwakeword/openwakeword/resources/models"
os.makedirs(models_dir, exist_ok=True)
for f in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    if not os.path.exists(f"{models_dir}/{f}"):
        !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f} -O {models_dir}/{f}
# Everything below that uses these packages runs in a fresh Python process, so no runtime restart is needed.

# ---- Download the 2 feature files training uses (-c resumes/skips finished downloads) ----
!wget -c -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -c -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
for d in ["./mit_rirs", "./audioset_16k", "./fma"]:
    os.makedirs(d, exist_ok=True)  # train.py lists these folders on start-up; training itself doesn't use them

# ---- Restore features + test clips from Google Drive ----
run_dir = "my_custom_model/hey_karl"
os.makedirs(run_dir, exist_ok=True)
for f in feature_names:
    src, dst = os.path.join(ckpt_dir, f), os.path.join(run_dir, f)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copyfile(src, dst)
    print(f"Restored {f} ({os.path.getsize(dst)/1e6:.0f} MB)")
info = json.load(open(os.path.join(ckpt_dir, "positive_test.tar.json")))
pt_dir = os.path.join(run_dir, "positive_test")
if not (os.path.isdir(pt_dir) and len(os.listdir(pt_dir)) == info["files"]):
    assert os.path.getsize(os.path.join(ckpt_dir, "positive_test.tar")) == info["bytes"], "positive_test.tar on Drive is incomplete"
    shutil.rmtree(pt_dir, ignore_errors=True)
    subprocess.run(["tar", "-xf", os.path.join(ckpt_dir, "positive_test.tar"), "-C", run_dir], check=True)
assert len(os.listdir(pt_dir)) == info["files"], f"got {len(os.listdir(pt_dir))} positive_test clips, expected {info['files']}"
print(f"Restored {info['files']} positive_test clips")
check = subprocess.run([sys.executable, "-c", f"""
import numpy as np
for f in {[os.path.join(run_dir, f) for f in feature_names]!r}:
    a = np.load(f, mmap_mode="r")
    assert a.ndim == 3 and a.shape[0] > 0 and bool(np.any(a[-1])), f + " is incomplete"
    print(f, a.shape)
"""], capture_output=True, text=True)
print(check.stdout, check.stderr)
assert check.returncode == 0, "A restored feature file is incomplete - see above"

# ---- Patch: train.py's --convert_to_tflite flag defaults to the string "False" (truthy) and crashes after
#      training. Skip it; the conversion is done below with onnx2tf.
train_py = "openwakeword/openwakeword/train.py"
src = open(train_py).read().replace('if args.convert_to_tflite:', 'if args.convert_to_tflite is True:')
assert 'if args.convert_to_tflite is True:' in src, "train.py has changed upstream; the patch didn't apply"
open(train_py, "w").write(src)

# ---- Training config (same as cell 3) ----
import yaml
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config.update({
    "target_phrase": ["hey_karl"], "adversarial_phrase": "hey karl", "model_name": "hey_karl",
    "n_samples": 50000, "n_samples_val": 5000, "augmentation_rounds": 2, "tts_batch_size": 100,
    "steps": number_of_training_steps, "max_negative_weight": false_activation_penalty, "layer_size": layer_size,
    "target_accuracy": 0.5, "target_recall": 0.25, "output_dir": "./my_custom_model",
    "rir_paths": ["./mit_rirs"], "background_paths": ["./audioset_16k", "./fma"],
    "background_paths_duplication_rate": [1, 1],
    "false_positive_validation_data_path": "validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
})
with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

# ---- Train ----
onnx_model_path = "my_custom_model/hey_karl.onnx"
tflite_path = "my_custom_model/hey_karl.tflite"
for f in glob.glob("my_custom_model/hey_karl*.onnx") + glob.glob("my_custom_model/hey_karl*.tflite"):
    os.remove(f)  # never convert/download a model left over from an earlier run
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model
assert os.path.exists(onnx_model_path), "Training did not produce a model - scroll up to the first error"
os.makedirs(drive_dir, exist_ok=True)
shutil.copy(onnx_model_path, drive_dir)  # save to Google Drive right away
print("Saved", onnx_model_path, "to Google Drive ->", drive_dir)

# ---- Convert ONNX -> TFLite with onnx2tf, then check the .tflite gives the same answers as the .onnx ----
convert = f"""
import os, subprocess, numpy as np, onnx, onnxruntime as ort
onnx_model_path, tflite_path = {onnx_model_path!r}, {tflite_path!r}
float32_path = tflite_path.replace(".tflite", "_float32.tflite")
kat_name = onnx.load(onnx_model_path).graph.input[0].name.replace(":", "__")  # e.g. "onnx____Flatten_0"
subprocess.run(["onnx2tf", "-i", onnx_model_path, "-o", os.path.dirname(onnx_model_path) + "/", "-kat", kat_name], check=True)
assert os.path.exists(float32_path), "onnx2tf did not produce a .tflite file - scroll up for the error"
from ai_edge_litert.interpreter import Interpreter
sess = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
interp = Interpreter(model_path=float32_path); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
assert list(inp["shape"]) == list(sess.get_inputs()[0].shape), f"TFLite input shape {{inp['shape']}} != ONNX {{sess.get_inputs()[0].shape}}"
x = np.random.rand(*inp["shape"]).astype(np.float32)
interp.set_tensor(inp["index"], x); interp.invoke()
diff = np.abs(interp.get_tensor(out["index"]).ravel() - sess.run(None, {{sess.get_inputs()[0].name: x}})[0].ravel()).max()
assert diff < 1e-3, f"TFLite and ONNX outputs differ by {{diff}}"
print(f"Model OK: input shape {{[int(d) for d in inp['shape']]}}, ONNX vs TFLite max difference {{diff:.1e}}")
os.replace(float32_path, tflite_path)  # only a checked model gets the final name
"""
open("convert_check.py", "w").write(convert)
!{sys.executable} convert_check.py
assert os.path.exists(tflite_path), "TFLite conversion/check failed - scroll up for the error"

# ---- Save to Drive + download ----
shutil.copy(tflite_path, drive_dir)
print("Saved", tflite_path, "to Google Drive ->", drive_dir)
files.download(onnx_model_path)
files.download(tflite_path)